(a) Sample 10 documents from TinyStories and OpenWebText. Using your previously-trained
TinyStories and OpenWebText tokenizers (10K and 32K vocabulary size, respectively),
encode these sampled documents into integer IDs. What is each tokenizer’s compression ratio
(bytes/token)?

10k的vocab在TinyStroiesV2-GPT-4的压缩率为4.075652359396108
32k的vocab在TinyStroiesV2-GPT-4的压缩率为3.8516939924711444

(b) What happens if you tokenize your OpenWebText sample with the TinyStories tokenizer?
Compare the compression ratio and/or qualitatively describe what happens

10k的vocab的tokenizer在TinyStroiesV2-GPT-4的压缩率为3.2889821401156545

(c) Estimate the throughput of your tokenizer (e.g., in bytes/second). How long would it take to
tokenize the Pile dataset (825GB of text)?

10k-vocab的bpe_tokenizer将大小为22502601B(约22MB)的文件tokenize,用时25.528261887000554s

tokenize 825GB text 预估用时1004945.11935445s (约280h)

(d) Using your TinyStories and OpenWebText tokenizers, encode the respective training and
development datasets into a sequence of integer token IDs. We’ll use this later to train our
language model. We recommend serializing the token IDs as a NumPy array of datatype
uint16. Why is uint16 an appropriate choice?

In [ ]:
import numpy as np
import os

def tokenize_and_save(dataset_path, tokenizer, output_path, max_length=None):
    vocab_size = len(tokenizer.id2token)
    assert vocab_size <= 65535,  f"Vocabulary {vocab_size} exceeds uint16 limit"
    print(f"✓ Vocabulary size: {vocab_size} (< 65,535)")

    f = open(dataset_path, "r", encoding='utf-8')

    all_token_ids = []

    for id in tokenizer.encode_iterable(f):
        all_token_ids.append(id)

    # Convert to uint16 numpy array
    token_array = np.array(all_token_ids, dtype=np.uint16)

    # Save to disk
    os.makedirs(output_path, exist_ok=True)
    output_path = os.path.join(output_path, "tokens.npy")
    np.save(output_path, token_array)
    
    print(f"✓ Saved {len(token_array):,} tokens to {output_path}")
    print(f"  File size: {os.path.getsize(output_path) / 1e6:.2f} MB")
    print(f"  dtype: {token_array.dtype}")

    return token_array

使用uint16保存token可以节约内存空间
1. uint16可以表示0 - 65535,我们的vocab大小约为10000-32000
2. 相比于uint32和uint64, 可以节省内存存储
 uint16: 2 bytes per token → 2 GB
 uint32: 4 bytes per token → 4 GB  
 uint64: 8 bytes per token → 8 GB
 int64:  8 bytes per token → 8 GB
3. 性能优势:
占用存储空间更小,小文件I/O更快
CPU cache可以容纳更多的token
减少内存带宽,更快地加载训练数据